# AlpCAN CXR Pipeline: Ark+ (Foundation Ark) Zero-Shot Baseline

**Amac:** NIH ChestX-ray14 tam veri seti (112,120 goruntu, 30,805 hasta) uzerinde Foundation Ark (Ark+) Swin-L modelinin zero-shot performansini degerlendirmek. AlpCAN CXR Pipeline Agent 7A baseline.

**Icindekiler:**
1. Veri seti yukleme — NIH ChestX-ray14 tam istatistikler
2. Ark+ agirliklari indirme (Dropbox)
3. ArkSwinTransformer model mimarisi tanimlama
4. Model yukleme ve dogrulama
5. Tam dataset zero-shot inference (112K+ goruntu, head_n=2)
6. AUC-ROC performans degerlendirmesi
7. DenseNet-121 ve Wang 2017 ile karsilastirma
8. Embedding analizi (UMAP/t-SNE)
9. Multi-model agreement analizi
10. Sonuclari kaydetme

**Model:** Ark+ (Foundation Ark) — Swin Transformer Large, 768x768 giris, 6 dataset uzerinde omni-label pre-trained

**Referans:** Haghighi et al., "Transferable Visual Words: Exploiting the Semantics of Anatomical Patterns for Self-Supervised Learning" IEEE TMI 2022

**GPU:** Kaggle T4 16GB

In [ ]:
# Kutuphaneleri kur — timm==0.5.4 ZORUNLU (sonraki surumler Swin mimarisini bozuyor)
!pip install -q timm==0.5.4 scikit-learn umap-learn

In [ ]:
import os
import time
import subprocess
import zipfile
import glob as glob_module
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import timm
import timm.models.swin_transformer as swin
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
assert timm.__version__.startswith('0.5.4'), f"timm 0.5.4 gerekli, mevcut: {timm.__version__}"
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. Veri Seti Yukleme — NIH ChestX-ray14

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Data_Entry CSV
csv_candidates = list(INPUT_DIR.rglob("Data_Entry*.csv"))
if not csv_candidates:
    raise FileNotFoundError("Data_Entry CSV bulunamadi! Dataset eklenmi\u015f mi?")
LABELS_CSV = csv_candidates[0]

# Goruntu klasorleri (images_001 - images_012)
all_img_dirs = sorted(INPUT_DIR.rglob("images"))
IMG_DIRS = [d for d in all_img_dirs if d.is_dir() and list(d.glob('*.png'))]

if not IMG_DIRS:
    # Dogrudan PNG'leri ara
    all_pngs = list(INPUT_DIR.rglob("*.png"))
    if all_pngs:
        IMG_DIRS = list(set(p.parent for p in all_pngs[:10]))

print(f"Labels CSV: {LABELS_CSV}")
print(f"Image directories: {len(IMG_DIRS)}")
for d in IMG_DIRS[:5]:
    n_files = len(list(d.glob('*.png')))
    print(f"  {d}: {n_files:,} PNG")

In [ ]:
# Tum goruntu yollarini index'le
image_index = {}
for img_dir in IMG_DIRS:
    for png in img_dir.glob('*.png'):
        image_index[png.name] = png

print(f"Toplam indekslenen goruntu: {len(image_index):,}")

In [ ]:
# Labels CSV'yi oku ve binary patoloji sutunlari olustur
df = pd.read_csv(LABELS_CSV)
print(f"CSV kayit sayisi: {len(df):,}")
print(f"Hasta sayisi: {df['Patient ID'].nunique():,}")
print(f"\nSutunlar: {list(df.columns)}")

# NIH14 etiket sirasi (Ark+ head_n=2 cikis sirasi ile uyumlu)
NIH14_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

# Alfabetik sirali etiketler (AUC hesaplama ve karsilastirma icin)
ALL_LABELS = sorted(NIH14_LABELS)

for label in NIH14_LABELS:
    df[label] = df['Finding Labels'].apply(lambda x: 1 if label in str(x) else 0)

df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if x == 'No Finding' else 0)

# Patoloji dagilimi
print("=" * 65)
print("NIH ChestX-ray14 — Patoloji Dagilimi (TAM DATASET)")
print("=" * 65)

label_counts = df[NIH14_LABELS].sum().sort_values(ascending=False)
for label, count in label_counts.items():
    pct = count / len(df) * 100
    marker = " << AlpCAN" if label in ('Nodule', 'Mass') else ""
    print(f"  {label:>20s}: {count:>6,} ({pct:>5.1f}%){marker}")

no_finding = df['No Finding'].sum()
print(f"  {'No Finding':>20s}: {no_finding:>6,} ({no_finding/len(df)*100:>5.1f}%)")
print(f"\n  Toplam goruntu: {len(df):,}")
print(f"  Toplam hasta:   {df['Patient ID'].nunique():,}")
print(f"  Multi-label:    {(df[NIH14_LABELS].sum(axis=1) > 1).sum():,} goruntu birden fazla patoloji iceriyor")

---
## 2. Ark+ Agirliklarini Indir

In [ ]:
# Ark+ agirliklari — Kaggle dataset'ten veya Dropbox'tan yukle
CHECKPOINT_NAME = "Ark6_swinLarge768_ep50.pth.tar"
CHECKPOINT_PATH = None
WEIGHTS_DIR = Path("/tmp/ark_weights")

# Oncelik 1: Kaggle dataset olarak yuklenmiş agirliklari ara
kaggle_weight_candidates = list(INPUT_DIR.rglob(CHECKPOINT_NAME))
if kaggle_weight_candidates:
    CHECKPOINT_PATH = kaggle_weight_candidates[0]
    print(f"Checkpoint Kaggle dataset'ten bulundu: {CHECKPOINT_PATH}")
    print(f"  Boyut: {CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")
else:
    # Oncelik 2: Dropbox'tan indir (yedek plan)
    WEIGHTS_ZIP = Path("/tmp/ark_weights.zip")
    
    # Daha once indirildiyseatla
    local_candidates = list(WEIGHTS_DIR.rglob(CHECKPOINT_NAME)) if WEIGHTS_DIR.exists() else []
    
    if local_candidates:
        CHECKPOINT_PATH = local_candidates[0]
        print(f"Checkpoint onceden indirilmis: {CHECKPOINT_PATH}")
    else:
        DROPBOX_URL = "https://www.dropbox.com/scl/fo/joycn8m93nvlrc8yjme40/AIHVIiLhTeCGx6_eI3GZLvg?rlkey=p3lphqvtgmiphw1n0u039jpjn&st=akcclipo&dl=1"
        
        print("Ark+ agirlikları Dropbox'tan indiriliyor...")
        print("UYARI: Bu islem ~20 dakika surebilir (21GB ZIP dosyasi)")
        start_dl = time.time()
        
        result = subprocess.run(
            ["wget", "-q", "--show-progress", DROPBOX_URL, "-O", str(WEIGHTS_ZIP)],
            capture_output=True, text=True, timeout=1800  # 30 dk timeout
        )
        
        if result.returncode != 0:
            print(f"wget hatasi: {result.stderr}")
            raise RuntimeError("Ark+ agirlikları indirilemedi!")
        
        dl_time = time.time() - start_dl
        zip_size = WEIGHTS_ZIP.stat().st_size / 1e9
        print(f"Indirme tamamlandi: {zip_size:.2f} GB ({dl_time/60:.1f} dk)")
        
        # ZIP'ten sadece gerekli checkpoint'i cikar
        print(f"ZIP'ten {CHECKPOINT_NAME} cikariliyor...")
        WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
        
        import zipfile
        with zipfile.ZipFile(str(WEIGHTS_ZIP), 'r') as zf:
            # Sadece bizim checkpoint dosyamizi ve fine-tuned versiyonu cikar
            target_files = [f for f in zf.namelist() 
                          if CHECKPOINT_NAME in f or 'ChestX-ray14' in f]
            print(f"Cikarilacak dosyalar ({len(target_files)}):")
            for tf in target_files:
                info = zf.getinfo(tf)
                print(f"  {tf} ({info.file_size / 1e9:.2f} GB)")
                zf.extract(tf, str(WEIGHTS_DIR))
        
        # Checkpoint yolunu bul
        local_candidates = list(WEIGHTS_DIR.rglob(CHECKPOINT_NAME))
        if local_candidates:
            CHECKPOINT_PATH = local_candidates[0]
        
        # ZIP sil (disk tasarrufu)
        WEIGHTS_ZIP.unlink(missing_ok=True)
        print("ZIP silindi (disk tasarrufu).")

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        f"{CHECKPOINT_NAME} bulunamadi!\n"
        "Cozum 1: 'ridvancebec/ark-plus-pretrained-weights-core' dataset'ini notebook'a ekleyin.\n"
        "Cozum 2: Internet baglantisini aktif edin (Dropbox indirmesi icin)."
    )

print(f"\nKullanilacak checkpoint: {CHECKPOINT_PATH}")
print(f"Boyut: {CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")

# Fine-tuned versiyon da var mi?
ft_name = "Ark+SwinL768_ChestX-ray14_ft.pth.tar"
ft_candidates = list(INPUT_DIR.rglob(ft_name))
if not ft_candidates and WEIGHTS_DIR.exists():
    ft_candidates = list(WEIGHTS_DIR.rglob(ft_name))
if ft_candidates:
    FT_CHECKPOINT_PATH = ft_candidates[0]
    print(f"\nFine-tuned checkpoint da mevcut: {FT_CHECKPOINT_PATH}")
    print(f"  Boyut: {FT_CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")
else:
    FT_CHECKPOINT_PATH = None
    print("\nFine-tuned checkpoint bulunamadi (opsiyonel).")

---
## 3. Model Mimarisi — ArkSwinTransformer

In [ ]:
class ArkSwinTransformer(swin.SwinTransformer):
    """Ark+ (Foundation Ark) Swin Transformer Large modeli.
    
    6 dataset uzerinde omni-label pre-trained:
      - MIMIC-CXR (14 sinif)
      - CheXpert (14 sinif)
      - NIH ChestX-ray14 (14 sinif) -> head_n=2
      - RSNA Pneumonia (3 sinif)
      - VinDr-CXR (6 sinif)
      - Shenzhen TB (1 sinif)
    
    Projector: Linear(1536 -> 1376), MLP yok
    """
    def __init__(self, num_classes_list, projector_features=None, use_mlp=False, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert num_classes_list is not None
        self.projector = None
        if projector_features:
            encoder_features = self.num_features
            self.num_features = projector_features
            if use_mlp:
                self.projector = nn.Sequential(
                    nn.Linear(encoder_features, self.num_features),
                    nn.ReLU(inplace=True),
                    nn.Linear(self.num_features, self.num_features)
                )
            else:
                self.projector = nn.Linear(encoder_features, self.num_features)
        self.omni_heads = []
        for num_classes in num_classes_list:
            self.omni_heads.append(nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity())
        self.omni_heads = nn.ModuleList(self.omni_heads)

    def forward(self, x, head_n=None):
        x = self.forward_features(x)
        if self.projector:
            x = self.projector(x)
        if head_n is not None:
            return x, self.omni_heads[head_n](x)
        else:
            return [head(x) for head in self.omni_heads]

    def generate_embeddings(self, x, after_proj=True):
        x = self.forward_features(x)
        if after_proj and self.projector:
            x = self.projector(x)
        return x

print("ArkSwinTransformer sinifi tanimlandi.")

In [ ]:
# Ark+ modelini olustur ve agirliklarini yukle
# Swin-L mimari parametreleri
NUM_CLASSES_LIST = [14, 14, 14, 3, 6, 1]  # MIMIC, CheXpert, NIH14, RSNA, VinDr, Shenzhen

model = ArkSwinTransformer(
    num_classes_list=NUM_CLASSES_LIST,
    projector_features=1376,
    use_mlp=False,
    img_size=768,
    patch_size=4,
    window_size=12,
    embed_dim=192,
    depths=(2, 2, 18, 2),
    num_heads=(6, 12, 24, 48),
    num_classes=0,  # Swin base sinif kafasi kullanilmiyor, omni_heads var
)

# Checkpoint yukle
print(f"Checkpoint yukleniyor: {CHECKPOINT_PATH}")
checkpoint = torch.load(str(CHECKPOINT_PATH), map_location='cpu')

# Checkpoint anahtarlarini kontrol et
print(f"Checkpoint anahtarlari: {list(checkpoint.keys())}")

# Teacher agirliklari al
state_dict = checkpoint['teacher']
print(f"Teacher state_dict anahtar sayisi: {len(state_dict)}")

# 'module.' on-ekini kaldir (DDP ile egitilmis modellerde olur)
cleaned_state_dict = {}
for k, v in state_dict.items():
    new_key = k.replace('module.', '') if k.startswith('module.') else k
    cleaned_state_dict[new_key] = v

print(f"Temizlenmis anahtar sayisi: {len(cleaned_state_dict)}")

# Ornek anahtarlar
sample_keys = list(cleaned_state_dict.keys())[:10]
print(f"Ornek anahtarlar: {sample_keys}")

# Agirlik yukle (strict=False: bazi anahtarlar uyusmayabilir)
load_result = model.load_state_dict(cleaned_state_dict, strict=False)
print(f"\nYukleme sonucu:")
print(f"  Eksik anahtarlar: {len(load_result.missing_keys)}")
print(f"  Beklenmeyen anahtarlar: {len(load_result.unexpected_keys)}")
if load_result.missing_keys:
    print(f"  Eksik (ilk 5): {load_result.missing_keys[:5]}")
if load_result.unexpected_keys:
    print(f"  Beklenmeyen (ilk 5): {load_result.unexpected_keys[:5]}")

# GPU'ya tasi
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# Model parametreleri
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel ozeti:")
print(f"  Mimari: Swin Transformer Large (768x768)")
print(f"  Toplam parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Projector: Linear(1536 -> 1376)")
print(f"  Omni-heads: {NUM_CLASSES_LIST}")
print(f"  NIH14 head (head_n=2): 14 sinif")
print(f"  Device: {device}")

# Hizli test — dummy input
with torch.no_grad():
    dummy = torch.randn(1, 3, 768, 768).to(device)
    emb, logits = model(dummy, head_n=2)
    print(f"\nTest ciktisi:")
    print(f"  Embedding shape: {emb.shape}")
    print(f"  Logits shape: {logits.shape}")
    print(f"  Logits range: [{logits.min().item():.3f}, {logits.max().item():.3f}]")

---
## 4. Zero-Shot Inference

In [ ]:
# NIH14 Dataset sinifi — 768x768 giris, ImageNet normalizasyonu
class NIH14Dataset(Dataset):
    """NIH ChestX-ray14 icin Ark+ uyumlu dataset.
    
    - 768x768 resize
    - 3-kanalli RGB'ye cevir (Swin-L icin)
    - ImageNet normalizasyonu: mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]
    """
    def __init__(self, dataframe, image_index, labels, img_size=768):
        self.df = dataframe.reset_index(drop=True)
        self.image_index = image_index
        self.labels = labels
        self.img_size = img_size
        
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),  # [0,255] -> [0,1], (H,W,C) -> (C,H,W)
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['Image Index']
        img_path = self.image_index.get(img_name)
        
        # Goruntu oku ve 3 kanala cevir (RGB)
        try:
            img = Image.open(img_path).convert('RGB')
            img = self.transform(img)
        except Exception as e:
            # Hata durumunda siyah goruntu dondur
            img = torch.zeros(3, self.img_size, self.img_size)
        
        # Etiketler
        label = torch.tensor([row[l] for l in self.labels], dtype=torch.float32)
        
        return img, label, img_name


print("NIH14Dataset sinifi tanimlandi.")
print(f"  Giris boyutu: 768x768x3")
print(f"  Normalizasyon: ImageNet (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])")
print(f"  Etiket sayisi: {len(NIH14_LABELS)}")

In [ ]:
# Train/test bolme — resmi NIH test listesini kullan (varsa)
# Resmi test listesi: test_list.txt (hasta bazli bolme)
test_list_candidates = list(INPUT_DIR.rglob("test_list*.txt"))

if test_list_candidates:
    test_list_path = test_list_candidates[0]
    print(f"Resmi test listesi bulundu: {test_list_path}")
    
    with open(test_list_path, 'r') as f:
        test_image_names = set(line.strip() for line in f if line.strip())
    
    # Mevcut goruntulerle filtrele
    df['exists'] = df['Image Index'].apply(lambda x: x in image_index)
    eval_df = df[df['exists']].reset_index(drop=True)
    
    test_mask = eval_df['Image Index'].isin(test_image_names)
    test_df = eval_df[test_mask].reset_index(drop=True)
    train_df = eval_df[~test_mask].reset_index(drop=True)
    
    print(f"  Train: {len(train_df):,} goruntu")
    print(f"  Test:  {len(test_df):,} goruntu")
    print(f"  Test hasta sayisi: {test_df['Patient ID'].nunique():,}")
else:
    print("Resmi test listesi bulunamadi. Hasta bazli 80/20 bolme yapiliyor...")
    
    df['exists'] = df['Image Index'].apply(lambda x: x in image_index)
    eval_df = df[df['exists']].reset_index(drop=True)
    
    # Hasta bazli bolme (veri sizintisini onle)
    np.random.seed(42)
    patient_ids = eval_df['Patient ID'].unique()
    np.random.shuffle(patient_ids)
    split_idx = int(len(patient_ids) * 0.8)
    train_patients = set(patient_ids[:split_idx])
    test_patients = set(patient_ids[split_idx:])
    
    train_df = eval_df[eval_df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    test_df = eval_df[eval_df['Patient ID'].isin(test_patients)].reset_index(drop=True)
    
    print(f"  Train: {len(train_df):,} goruntu ({len(train_patients):,} hasta)")
    print(f"  Test:  {len(test_df):,} goruntu ({len(test_patients):,} hasta)")

# Tam dataset icin (inference tum goruntulerde yapilacak)
full_df = eval_df.copy()
print(f"\nTam dataset: {len(full_df):,} goruntu")
print(f"  Nodule: {full_df['Nodule'].sum():,}")
print(f"  Mass: {full_df['Mass'].sum():,}")

In [ ]:
# TAM DATASET uzerinde zero-shot inference
# 768x768 cozunurluk buyuk — batch_size=16, AMP kullan
BATCH_SIZE = 16
HEAD_N = 2  # NIH14 head

full_dataset = NIH14Dataset(full_df, image_index, NIH14_LABELS, img_size=768)
full_loader = DataLoader(
    full_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

all_predictions = []
all_labels = []
all_image_names = []
n_processed = 0
start_time = time.time()

print(f"Ark+ Zero-Shot Inference basliyor")
print(f"  Dataset: {len(full_dataset):,} goruntu")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Head: NIH14 (head_n={HEAD_N})")
print(f"  Cozunurluk: 768x768")
print(f"  AMP: Etkin")
print("-" * 60)

with torch.no_grad():
    for batch_idx, (images, labels, names) in enumerate(full_loader):
        images = images.to(device)
        
        # AMP ile inference (bellek tasarrufu)
        with torch.cuda.amp.autocast():
            _, logits = model(images, head_n=HEAD_N)
        
        # Sigmoid uygula — multi-label tahmin
        probs = torch.sigmoid(logits).cpu().numpy()
        
        all_predictions.append(probs)
        all_labels.append(labels.numpy())
        all_image_names.extend(names)
        
        n_processed += len(images)
        
        # Ilerleme raporu — her 500 goruntuye bir
        if n_processed % 500 < BATCH_SIZE or batch_idx == len(full_loader) - 1:
            elapsed = time.time() - start_time
            speed = n_processed / elapsed if elapsed > 0 else 0
            remaining = len(full_dataset) - n_processed
            eta = remaining / speed if speed > 0 else 0
            print(f"  {n_processed:>7,}/{len(full_dataset):,} "
                  f"({n_processed/len(full_dataset)*100:>5.1f}%) "
                  f"| {speed:.0f} img/s | ETA: {eta/60:.1f} min")

# Birlestir
predictions = np.concatenate(all_predictions, axis=0)
gt_labels = np.concatenate(all_labels, axis=0)
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"Ark+ Zero-Shot Inference tamamlandi!")
print(f"   Islenen: {predictions.shape[0]:,} goruntu")
print(f"   Sure: {elapsed/60:.1f} dakika ({predictions.shape[0]/elapsed:.0f} img/s)")
print(f"   Prediction shape: {predictions.shape}")
print(f"   GT shape: {gt_labels.shape}")
print(f"   Prediction range: [{predictions.min():.4f}, {predictions.max():.4f}]")

---
## 5. AUC-ROC Performans Degerlendirmesi

In [ ]:
# Wang et al. 2017 orijinal sonuclari (referans baseline)
WANG_AUC = {
    'Atelectasis': 0.716, 'Cardiomegaly': 0.814, 'Effusion': 0.784,
    'Infiltration': 0.609, 'Mass': 0.706, 'Nodule': 0.671,
    'Pneumonia': 0.633, 'Pneumothorax': 0.806, 'Consolidation': 0.708,
    'Edema': 0.805, 'Emphysema': 0.833, 'Fibrosis': 0.786,
    'Pleural_Thickening': 0.684, 'Hernia': 0.872
}

# DenseNet-121 sonuclarini yukle (varsa)
DENSENET_AUC = {}
densenet_auc_paths = [
    INPUT_DIR / 'alpcan-cxr-pipeline-torchxrayvision-baseline' / 'cxr_auc_scores.csv',
    Path('/kaggle/input/alpcan-cxr-pipeline-torchxrayvision-baseline/cxr_auc_scores.csv'),
    OUTPUT_DIR / 'cxr_auc_scores.csv',
]

densenet_loaded = False
for p in densenet_auc_paths:
    if p.exists():
        densenet_auc_df = pd.read_csv(p)
        for _, row in densenet_auc_df.iterrows():
            DENSENET_AUC[row['pathology']] = row['auc_roc']
        print(f"DenseNet-121 AUC sonuclari yuklendi: {p}")
        densenet_loaded = True
        break

if not densenet_loaded:
    print("DenseNet-121 AUC sonuclari bulunamadi.")
    print("Karsilastirma Wang 2017 ile yapilacak.")

# Per-patoloji AUC-ROC hesapla
print("\n" + "=" * 75)
print("Ark+ (Foundation Ark) Zero-Shot — NIH ChestX-ray14 Performansi")
print(f"N = {len(predictions):,} goruntu")
print("=" * 75)
print(f"{'Patoloji':>20s} | {'Ark+ AUC':>9s} | {'Wang2017':>8s} | {'D121 AUC':>9s} | {'N_pos':>7s} | {'Prev':>6s}")
print("-" * 75)

auc_results = {}
ap_results = {}

for i, label in enumerate(NIH14_LABELS):
    y_true = gt_labels[:, i]
    y_score = predictions[:, i]
    
    n_pos = int(y_true.sum())
    prevalence = n_pos / len(y_true) * 100
    
    if n_pos > 0 and n_pos < len(y_true):
        auc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        auc_results[label] = auc
        ap_results[label] = ap
        
        wang_str = f"{WANG_AUC.get(label, 0):>8.4f}" if label in WANG_AUC else "     N/A"
        d121_str = f"{DENSENET_AUC.get(label, 0):>9.4f}" if label in DENSENET_AUC else "      N/A"
        marker = " <<" if label in ('Nodule', 'Mass') else ""
        print(f"  {label:>20s} | {auc:>9.4f} | {wang_str} | {d121_str} | {n_pos:>7,} | {prevalence:>5.1f}%{marker}")
    else:
        print(f"  {label:>20s} | {'N/A':>9s} | {'':>8s} | {'':>9s} | {n_pos:>7,} | {prevalence:>5.1f}%")

print("-" * 75)
if auc_results:
    mean_auc = np.mean(list(auc_results.values()))
    mean_ap = np.mean(list(ap_results.values()))
    print(f"  {'Ortalama':>20s} | {mean_auc:>9.4f} |")
    
    cancer_labels = [l for l in ('Nodule', 'Mass') if l in auc_results]
    if cancer_labels:
        cancer_auc = np.mean([auc_results[l] for l in cancer_labels])
        print(f"  {'Nodule+Mass ort.':>20s} | {cancer_auc:>9.4f} |                         <- AlpCAN odak")

---
## 6. DenseNet-121 ve Wang 2017 ile Karsilastirma

In [ ]:
# Detayli karsilastirma tablosu
print("=" * 85)
print("Ark+ vs DenseNet-121 vs Wang et al. 2017 (CVPR)")
print("=" * 85)

comparison_data = []

if DENSENET_AUC:
    print(f"{'Patoloji':>20s} | {'Ark+':>8s} | {'DenseNet':>8s} | {'Wang':>8s} | {'Ark-D121':>8s} | {'Ark-Wang':>8s}")
    print("-" * 85)
    
    for label in sorted(auc_results.keys()):
        ark = auc_results[label]
        d121 = DENSENET_AUC.get(label, None)
        wang = WANG_AUC.get(label, None)
        
        diff_d121 = f"{ark - d121:>+7.4f}" if d121 is not None else "    N/A"
        diff_wang = f"{ark - wang:>+7.4f}" if wang is not None else "    N/A"
        d121_str = f"{d121:>8.4f}" if d121 is not None else "     N/A"
        wang_str = f"{wang:>8.4f}" if wang is not None else "     N/A"
        
        marker = " *" if label in ('Nodule', 'Mass') else ""
        print(f"  {label:>20s} | {ark:>8.4f} | {d121_str} | {wang_str} | {diff_d121} | {diff_wang}{marker}")
        
        comparison_data.append({
            'pathology': label,
            'ark_plus_auc': ark,
            'densenet121_auc': d121,
            'wang_2017_auc': wang,
            'ark_vs_d121': ark - d121 if d121 else None,
            'ark_vs_wang': ark - wang if wang else None,
        })
    
    print("-" * 85)
    ark_vals = [auc_results[l] for l in sorted(auc_results.keys())]
    d121_vals = [DENSENET_AUC.get(l, 0) for l in sorted(auc_results.keys()) if l in DENSENET_AUC]
    if d121_vals:
        print(f"  {'Ort. fark (Ark-D121)':>20s} | {np.mean([c['ark_vs_d121'] for c in comparison_data if c['ark_vs_d121'] is not None]):>+7.4f}")
else:
    print(f"{'Patoloji':>20s} | {'Ark+':>8s} | {'Wang':>8s} | {'Fark':>8s}")
    print("-" * 55)
    
    for label in sorted(auc_results.keys()):
        ark = auc_results[label]
        wang = WANG_AUC.get(label, 0)
        diff = ark - wang
        arrow = ">>" if diff > 0 else "<<"
        print(f"  {label:>20s} | {ark:>8.4f} | {wang:>8.4f} | {diff:>+7.4f} {arrow}")
        
        comparison_data.append({
            'pathology': label,
            'ark_plus_auc': ark,
            'densenet121_auc': None,
            'wang_2017_auc': wang,
            'ark_vs_d121': None,
            'ark_vs_wang': ark - wang,
        })
    
    print("-" * 55)
    mean_diff = np.mean([c['ark_vs_wang'] for c in comparison_data if c['ark_vs_wang'] is not None])
    print(f"  {'Ort. fark (Ark-Wang)':>20s} | {'':>8s} | {'':>8s} | {mean_diff:>+7.4f}")

In [ ]:
# Karsilastirma gorseli — bar chart
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 1. AUC karsilastirma — yatay bar chart
labels_sorted = sorted(auc_results.keys(), key=lambda x: auc_results[x], reverse=True)
ark_vals = [auc_results[l] for l in labels_sorted]
wang_vals = [WANG_AUC.get(l, 0) for l in labels_sorted]
y_pos = np.arange(len(labels_sorted))

if DENSENET_AUC:
    d121_vals = [DENSENET_AUC.get(l, 0) for l in labels_sorted]
    bar_w = 0.25
    axes[0].barh(y_pos + bar_w, wang_vals, bar_w, label='Wang 2017', color='#bdc3c7', alpha=0.8)
    axes[0].barh(y_pos, d121_vals, bar_w, label='DenseNet-121', color='#3498db')
    axes[0].barh(y_pos - bar_w, ark_vals, bar_w, label='Ark+ (Swin-L)', color='#e74c3c')
else:
    bar_w = 0.3
    axes[0].barh(y_pos + 0.15, wang_vals, bar_w, label='Wang 2017', color='#bdc3c7', alpha=0.8)
    axes[0].barh(y_pos - 0.15, ark_vals, bar_w, label='Ark+ (Swin-L)', color='#e74c3c')

# Nodule ve Mass vurgula
for i, l in enumerate(labels_sorted):
    if l in ('Nodule', 'Mass'):
        offset = -bar_w if DENSENET_AUC else -0.15
        axes[0].barh(i + offset, auc_results[l], bar_w, color='#c0392b', edgecolor='black', linewidth=1.5)

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels_sorted)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('AUC-ROC Karsilastirma', fontsize=13, fontweight='bold')
axes[0].set_xlabel('AUC-ROC')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].axvline(x=0.7, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.3)
axes[0].invert_yaxis()

# 2. ROC egrileri — AlpCAN odak patolojileri
for j, label in enumerate(['Nodule', 'Mass', 'Consolidation', 'Infiltration']):
    if label in auc_results:
        idx_l = NIH14_LABELS.index(label)
        y_true = gt_labels[:, idx_l]
        y_score = predictions[:, idx_l]
        if y_true.sum() > 0:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc_val = auc_results[label]
            lw = 3 if label in ('Nodule', 'Mass') else 1.5
            axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc_val:.3f})", linewidth=lw)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Egrileri — AlpCAN Odak Patolojileri', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.2)

# 3. Fark grafigi — Ark+ vs Wang 2017
diffs = [auc_results[l] - WANG_AUC.get(l, 0) for l in labels_sorted if l in WANG_AUC]
diff_labels = [l for l in labels_sorted if l in WANG_AUC]
diff_colors = ['#27ae60' if d > 0 else '#e74c3c' for d in diffs]
y_pos_d = np.arange(len(diff_labels))

axes[2].barh(y_pos_d, diffs, color=diff_colors)
axes[2].axvline(x=0, color='black', linewidth=0.5)
axes[2].set_yticks(y_pos_d)
axes[2].set_yticklabels(diff_labels)
axes[2].set_title('AUC Farki: Ark+ - Wang 2017', fontsize=13, fontweight='bold')
axes[2].set_xlabel('AUC Farki')
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ark_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cxr_ark_performance_comparison.png")

---
## 7. Embedding Analizi — UMAP/t-SNE

In [ ]:
# 2000 ornek goruntu icin 1376-dim embedding cikar
N_EMBEDDING_SAMPLES = 2000

# Dengeli ornekleme: Nodule, Mass, Normal ve Diger
nodule_mask = full_df['Nodule'] == 1
mass_mask = full_df['Mass'] == 1
normal_mask = full_df['No Finding'] == 1
other_mask = ~(nodule_mask | mass_mask | normal_mask)

n_per_group = N_EMBEDDING_SAMPLES // 4

np.random.seed(42)
sample_indices = []

for mask, name, target_n in [
    (nodule_mask, 'Nodule', n_per_group),
    (mass_mask, 'Mass', n_per_group),
    (normal_mask, 'Normal', n_per_group),
    (other_mask, 'Other', n_per_group),
]:
    available = np.where(mask)[0]
    n_sample = min(target_n, len(available))
    selected = np.random.choice(available, n_sample, replace=False)
    sample_indices.extend(selected.tolist())
    print(f"  {name}: {n_sample} ornek secildi (mevcut: {len(available):,})")

sample_df = full_df.iloc[sample_indices].reset_index(drop=True)
print(f"\nToplam embedding ornegi: {len(sample_df):,}")

# Embedding dataset
emb_dataset = NIH14Dataset(sample_df, image_index, NIH14_LABELS, img_size=768)
emb_loader = DataLoader(emb_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Embedding cikar
all_embeddings = []
all_emb_names = []
all_emb_labels = []

print("\nEmbedding cikariliyor...")
start_emb = time.time()

with torch.no_grad():
    for images, labels, names in emb_loader:
        images = images.to(device)
        with torch.cuda.amp.autocast():
            emb = model.generate_embeddings(images, after_proj=True)
        all_embeddings.append(emb.cpu().float().numpy())
        all_emb_names.extend(names)
        all_emb_labels.append(labels.numpy())

embeddings = np.concatenate(all_embeddings, axis=0)
emb_labels = np.concatenate(all_emb_labels, axis=0)
emb_time = time.time() - start_emb

print(f"Embedding tamamlandi: {embeddings.shape} ({emb_time:.1f}s)")
print(f"  Boyut: {embeddings.shape[1]} (projector sonrasi)")

In [ ]:
# UMAP ve t-SNE ile gorsellestirme
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn yuklu degil, sadece t-SNE kullanilacak.")

# Grup etiketleri olustur: Nodule, Mass, Normal, Other
nodule_idx = NIH14_LABELS.index('Nodule')
mass_idx = NIH14_LABELS.index('Mass')

group_labels = []
for i in range(len(emb_labels)):
    if emb_labels[i, nodule_idx] == 1:
        group_labels.append('Nodule')
    elif emb_labels[i, mass_idx] == 1:
        group_labels.append('Mass')
    elif emb_labels[i].sum() == 0:
        group_labels.append('Normal')
    else:
        group_labels.append('Other')

group_labels = np.array(group_labels)
print("Grup dagilimi:")
for g in ['Nodule', 'Mass', 'Normal', 'Other']:
    print(f"  {g}: {(group_labels == g).sum()}")

# Renk paleti
color_map = {'Nodule': '#e74c3c', 'Mass': '#e67e22', 'Normal': '#3498db', 'Other': '#95a5a6'}
colors = [color_map[g] for g in group_labels]

n_plots = 2 if HAS_UMAP else 1
fig, axes = plt.subplots(1, n_plots, figsize=(8 * n_plots, 7))
if n_plots == 1:
    axes = [axes]

# t-SNE
print("t-SNE calistiriliyor...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb_tsne = tsne.fit_transform(embeddings)

ax = axes[0]
for g in ['Other', 'Normal', 'Mass', 'Nodule']:  # Nodule en ustte olsun
    mask = group_labels == g
    alpha = 0.7 if g in ('Nodule', 'Mass') else 0.3
    s = 25 if g in ('Nodule', 'Mass') else 10
    ax.scatter(emb_tsne[mask, 0], emb_tsne[mask, 1], c=color_map[g], 
              label=f"{g} (n={mask.sum()})", alpha=alpha, s=s)
ax.set_title('t-SNE — Ark+ 1376-dim Embeddings', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(True, alpha=0.1)

# UMAP (varsa)
if HAS_UMAP:
    print("UMAP calistiriliyor...")
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    emb_umap = reducer.fit_transform(embeddings)
    
    ax = axes[1]
    for g in ['Other', 'Normal', 'Mass', 'Nodule']:
        mask = group_labels == g
        alpha = 0.7 if g in ('Nodule', 'Mass') else 0.3
        s = 25 if g in ('Nodule', 'Mass') else 10
        ax.scatter(emb_umap[mask, 0], emb_umap[mask, 1], c=color_map[g],
                  label=f"{g} (n={mask.sum()})", alpha=alpha, s=s)
    ax.set_title('UMAP — Ark+ 1376-dim Embeddings', fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.grid(True, alpha=0.1)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ark_embedding_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cxr_ark_embedding_visualization.png")

---
## 8. Multi-Model Agreement Analizi

In [ ]:
# DenseNet-121 tahminlerini yukle (kernel source'dan)
densenet_preds_df = None
densenet_preds_paths = [
    INPUT_DIR / 'alpcan-cxr-pipeline-torchxrayvision-baseline' / 'cxr_predictions_full.csv',
    Path('/kaggle/input/alpcan-cxr-pipeline-torchxrayvision-baseline/cxr_predictions_full.csv'),
    OUTPUT_DIR / 'cxr_predictions_full.csv',
]

for p in densenet_preds_paths:
    if p.exists():
        densenet_preds_df = pd.read_csv(p)
        print(f"DenseNet-121 tahminleri yuklendi: {p}")
        print(f"  Shape: {densenet_preds_df.shape}")
        print(f"  Sutunlar (ornek): {list(densenet_preds_df.columns[:10])}")
        break

if densenet_preds_df is None:
    print("DenseNet-121 tahminleri bulunamadi.")
    print("Agreement analizi icin kernel_sources'a 'ridvancebec/alpcan-cxr-pipeline-torchxrayvision-baseline' ekleyin.")
    print("\nSimulasyon modu ile devam ediliyor...")

In [ ]:
# Multi-model agreement — hangi goruntuler her iki model tarafindan tespit ediliyor?
# Ark+ tahminleri DataFrame'e cevir
ark_preds_df = pd.DataFrame({'image_name': all_image_names})
for i, label in enumerate(NIH14_LABELS):
    ark_preds_df[f'ark_{label}'] = predictions[:, i]

if densenet_preds_df is not None:
    # Ortak goruntuleri bul
    ark_names = set(all_image_names)
    d121_names = set(densenet_preds_df['image_name'].values)
    common_names = sorted(ark_names & d121_names)
    print(f"Ortak goruntu sayisi: {len(common_names):,}")
    
    # Nodule ve Mass icin agreement
    for label in ['Nodule', 'Mass']:
        ark_col = f'ark_{label}'
        
        # DenseNet-121 sutun ismi bul
        d121_col_candidates = [c for c in densenet_preds_df.columns 
                               if label.lower() in c.lower() and c.startswith('pred_')]
        if not d121_col_candidates:
            d121_col_candidates = [c for c in densenet_preds_df.columns 
                                   if label.lower() in c.lower()]
        
        if not d121_col_candidates:
            print(f"  {label}: DenseNet-121 sutunu bulunamadi, atlaniyor.")
            continue
        d121_col = d121_col_candidates[0]
        
        # Merge
        ark_sub = ark_preds_df[ark_preds_df['image_name'].isin(common_names)][['image_name', ark_col]]
        d121_sub = densenet_preds_df[densenet_preds_df['image_name'].isin(common_names)][['image_name', d121_col]]
        merged = ark_sub.merge(d121_sub, on='image_name')
        
        # Ground truth ekle
        gt_df = full_df[['Image Index', label]].rename(columns={'Image Index': 'image_name'})
        merged = merged.merge(gt_df, on='image_name', how='inner')
        
        # Youden J optimal threshold ile karsilastir
        # Basit threshold: median pozitif skor
        pos_mask = merged[label] == 1
        ark_threshold = 0.5  # sigmoid cikisi icin 0.5 dogal threshold
        d121_threshold = merged.loc[pos_mask, d121_col].median() if pos_mask.sum() > 0 else 0.5
        
        merged['ark_pred'] = (merged[ark_col] >= ark_threshold).astype(int)
        merged['d121_pred'] = (merged[d121_col] >= d121_threshold).astype(int)
        
        # Agreement matrisi
        both_pos = ((merged['ark_pred'] == 1) & (merged['d121_pred'] == 1)).sum()
        only_ark = ((merged['ark_pred'] == 1) & (merged['d121_pred'] == 0)).sum()
        only_d121 = ((merged['ark_pred'] == 0) & (merged['d121_pred'] == 1)).sum()
        both_neg = ((merged['ark_pred'] == 0) & (merged['d121_pred'] == 0)).sum()
        
        agreement = (both_pos + both_neg) / len(merged) * 100
        
        print(f"\n{label} Agreement Analizi:")
        print(f"  Threshold — Ark+: {ark_threshold:.3f}, DenseNet: {d121_threshold:.3f}")
        print(f"  Her iki model pozitif:  {both_pos:>7,}")
        print(f"  Sadece Ark+:            {only_ark:>7,}")
        print(f"  Sadece DenseNet-121:    {only_d121:>7,}")
        print(f"  Her iki model negatif:  {both_neg:>7,}")
        print(f"  Agreement:              {agreement:.1f}%")
        
        # True positive'lerde agreement
        if pos_mask.sum() > 0:
            tp_both = ((merged.loc[pos_mask, 'ark_pred'] == 1) & (merged.loc[pos_mask, 'd121_pred'] == 1)).sum()
            tp_only_ark = ((merged.loc[pos_mask, 'ark_pred'] == 1) & (merged.loc[pos_mask, 'd121_pred'] == 0)).sum()
            tp_only_d121 = ((merged.loc[pos_mask, 'ark_pred'] == 0) & (merged.loc[pos_mask, 'd121_pred'] == 1)).sum()
            tp_missed = ((merged.loc[pos_mask, 'ark_pred'] == 0) & (merged.loc[pos_mask, 'd121_pred'] == 0)).sum()
            
            print(f"\n  Gercek pozitifler icinde (N={pos_mask.sum():,}):")
            print(f"    Her ikisi yakaliyor:    {tp_both:>5,} ({tp_both/pos_mask.sum()*100:.1f}%)")
            print(f"    Sadece Ark+ yakaliyor:  {tp_only_ark:>5,} ({tp_only_ark/pos_mask.sum()*100:.1f}%)")
            print(f"    Sadece D121 yakaliyor:  {tp_only_d121:>5,} ({tp_only_d121/pos_mask.sum()*100:.1f}%)")
            print(f"    Her ikisi kaciyor:      {tp_missed:>5,} ({tp_missed/pos_mask.sum()*100:.1f}%)")
            print(f"    Ensemble avantaji:      {tp_only_ark + tp_only_d121:>5,} ek tespit (ensemble ile)")
else:
    print("\nDenseNet-121 tahminleri mevcut degil — agreement analizi atlandiyor.")
    print("Bu analiz icin Notebook 02 ciktilarini kernel_sources olarak ekleyin.")

---
## 9. Sonuclari Kaydet

In [ ]:
# 1. Tum tahminleri CSV
results_df = pd.DataFrame({'image_name': all_image_names})
for i, label in enumerate(NIH14_LABELS):
    results_df[f'pred_{label}'] = predictions[:, i]
for i, label in enumerate(NIH14_LABELS):
    results_df[f'gt_{label}'] = gt_labels[:, i]

results_df.to_csv(OUTPUT_DIR / 'cxr_ark_predictions_full.csv', index=False)
print(f"cxr_ark_predictions_full.csv ({len(results_df):,} rows, {results_df.shape[1]} cols)")

# 2. AUC sonuclari
auc_df = pd.DataFrame([
    {
        'pathology': k,
        'model': 'ark_plus_swin_large_768',
        'auc_roc': v,
        'ap': ap_results.get(k, 0),
        'n_positive': int(gt_labels[:, NIH14_LABELS.index(k)].sum()),
        'prevalence': gt_labels[:, NIH14_LABELS.index(k)].mean(),
        'wang_2017_auc': WANG_AUC.get(k, None),
        'improvement_vs_wang': v - WANG_AUC.get(k, 0) if k in WANG_AUC else None,
        'densenet121_auc': DENSENET_AUC.get(k, None),
        'improvement_vs_densenet': v - DENSENET_AUC.get(k, 0) if k in DENSENET_AUC else None,
    }
    for k, v in auc_results.items()
]).sort_values('auc_roc', ascending=False)

auc_df.to_csv(OUTPUT_DIR / 'cxr_ark_auc_scores.csv', index=False)
print(f"cxr_ark_auc_scores.csv")

# 3. Embeddings (numpy)
np.savez_compressed(
    OUTPUT_DIR / 'cxr_ark_embeddings.npz',
    embeddings=embeddings,
    image_names=np.array(all_emb_names),
    labels=emb_labels,
    label_names=np.array(NIH14_LABELS),
    group_labels=group_labels,
)
print(f"cxr_ark_embeddings.npz ({embeddings.shape[0]} ornekler, {embeddings.shape[1]} boyut)")

# 4. Karsilastirma tablosu
if comparison_data:
    comp_df = pd.DataFrame(comparison_data)
    comp_df.to_csv(OUTPUT_DIR / 'cxr_ark_comparison.csv', index=False)
    print(f"cxr_ark_comparison.csv")

# Cikti dosyalarini listele
print(f"\nKaydedilen dosyalar:")
for f in sorted(OUTPUT_DIR.glob('cxr_ark_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

---
## 10. Final Ozet

In [ ]:
# Kapsamli ozet
print("\n" + "=" * 75)
print("AlpCAN CXR Pipeline — Ark+ (Foundation Ark) Zero-Shot Baseline OZET")
print("=" * 75)

print(f"\n--- Dataset ---")
print(f"  NIH ChestX-ray14")
print(f"  Toplam goruntu: {len(full_df):,}")
print(f"  Toplam hasta:   {full_df['Patient ID'].nunique():,}")
print(f"  Islenen:        {predictions.shape[0]:,}")

print(f"\n--- Model ---")
print(f"  Ark+ (Foundation Ark) — Swin Transformer Large")
print(f"  Giris: 768x768x3, Patch: 4, Window: 12")
print(f"  Embed: 192, Depths: (2,2,18,2), Heads: (6,12,24,48)")
print(f"  Projector: Linear(1536 -> 1376)")
print(f"  Omni-heads: {NUM_CLASSES_LIST} (6 dataset)")
print(f"  NIH14 head: head_n=2 (zero-shot)")
print(f"  Parametre: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"  Checkpoint: {CHECKPOINT_NAME}")

if auc_results:
    print(f"\n--- Performans (AUC-ROC) ---")
    mean_auc = np.mean(list(auc_results.values()))
    print(f"  Ortalama AUC:  {mean_auc:.4f}")
    
    for label in NIH14_LABELS:
        if label in auc_results:
            wang = WANG_AUC.get(label, 0)
            diff = auc_results[label] - wang
            d121_str = ""
            if label in DENSENET_AUC:
                d121_diff = auc_results[label] - DENSENET_AUC[label]
                d121_str = f" | D121: {d121_diff:+.4f}"
            marker = " << AlpCAN" if label in ('Nodule', 'Mass') else ""
            print(f"  {label:>20s}: {auc_results[label]:.4f} (Wang fark: {diff:+.4f}{d121_str}){marker}")
    
    cancer_labels = [l for l in ('Nodule', 'Mass') if l in auc_results]
    if cancer_labels:
        cancer_auc = np.mean([auc_results[l] for l in cancer_labels])
        print(f"\n  Nodule+Mass ort. AUC: {cancer_auc:.4f}")

print(f"\n--- Embedding ---")
print(f"  Boyut: {embeddings.shape[1]}")
print(f"  Ornek: {embeddings.shape[0]:,} goruntu")

print(f"\n--- Cikti Dosyalari ---")
for f in sorted(OUTPUT_DIR.glob('cxr_ark_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. X-Raydar ResNet+Transformer baseline (Agent 7C)")
print(f"  2. 4 model gercek ensemble entegrasyonu")
print(f"  3. Ark+ embedding'leri ile downstream task fine-tuning")
print(f"  4. CXR pipeline sonuclarini LIDC-IDRI CT pipeline ile birlestirme")
print(f"  5. AlpCAN karar destek sistemi (ensemble + threshold optimizasyon)")
print("=" * 75)